In [ ]:
%load_ext autoreload
%autoreload 2

import nest_asyncio
nest_asyncio.apply()

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
plt.style.use('ggplot')
params = {'legend.fontsize': 'medium',
        'figure.figsize': (18, 8),
        'axes.labelsize': 'medium',
        'axes.titlesize': 'large',
        'xtick.labelsize': 'medium',
        'ytick.labelsize': 'medium'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import datetime
import pytz

NYC = pytz.timezone('America/New_York')

import sys
sys.path.append('../../')

# SFR Calendar Spread RV Backtest (Event-Driven)

Two-leg calendar spreads (e.g., SFR1/SFR2 = M26/U26) using the `QueryDrivenBacktest` framework.

**Strategy:** Enter spreads when z-score exceeds threshold (pay spread if cheap, receive if rich). Exit on mean-reversion, stop-loss, take-profit, or max holding period.

**Tweak `BACKTEST_CONFIG` below and re-run.**

---
## 1. Configuration

In [ ]:
BACKTEST_CONFIG = {
    # -- Data --
    'data_start': '2024-06-01',
    'bt_start':   '2025-01-02',
    'bt_end':     'live',
    'n_contracts': 12,
    'constant_maturity': True,
    'roll_adjusted': True,
    'source': 'BARCHART_STIRF-RL',
    'curve': 'USD-SOFR-1D-Q12STIRT',

    # -- Structure --
    'spread_gap': 1,                   # 1=3M spread, 2=6M, 3=9M, 4=12M

    # -- Windows --
    'zscore_window': 60,
    'vol_window': 20,

    # -- Entry --
    'entry_min_zscore': 1.5,
    'entry_require_carry': False,
    'entry_max_vol': None,

    # -- Exit (first match wins) --
    'exit_mean_reversion': True,
    'exit_take_profit_zscore': None,
    'exit_take_profit_bp': None,
    'exit_stop_loss_sd': 2.0,
    'exit_stop_loss_bp': None,
    'exit_max_holding_days': 22,

    # -- Portfolio --
    'max_concurrent_trades': 3,
    'no_duplicate_flies': True,

    # -- Sizing --
    'belly_bpv': 100_000,
    'round_trip_cost_bp': 0.5,
}

gap_label = {1: '3M', 2: '6M', 3: '9M', 4: '12M'}[BACKTEST_CONFIG['spread_gap']]
print(f'Config: {gap_label} calendar spreads | Z>{BACKTEST_CONFIG["entry_min_zscore"]} | MaxHold={BACKTEST_CONFIG["exit_max_holding_days"]}d')

---
## 2. Load Data & Compute Spread Signals

In [ ]:
from BT.signals.sfr_cal_spread_rv import (
    SFRCalSpreadRVConfig,
    StructureType,
    STRUCTURE_LABELS,
    load_rate_panel,
    compute_spread_curve,
    compute_zscore_ts,
    analyze_specific_spread,
    build_snapshot,
)
from BT.signals.sfr_fly_triggers import (
    build_spread_signal_table,
    SFRSpreadEntryTrigger,
    SFRSpreadExitTrigger,
)
from BT.data_handler import TimeGrid
from BT.query_engine import QueryDrivenBacktest
from BT.query_strategy import QueryStrategy
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.TimeseriesBuilder import TimeseriesBuilder

C = BACKTEST_CONFIG

sfr_config = SFRCalSpreadRVConfig(
    n_contracts=C['n_contracts'], zscore_window=C['zscore_window'],
    vol_window=C['vol_window'], constant_maturity=C['constant_maturity'],
    roll_adjusted=C['roll_adjusted'], source=C['source'], curve=C['curve'],
)

curve_mdp = IRSwapsMDP(source=sfr_config.source)
ts_builder = TimeseriesBuilder()

start = NYC.localize(datetime.datetime.fromisoformat(C['data_start']).replace(hour=18))

print('Loading rate panel...')
rates = load_rate_panel(sfr_config, start=start, end=C['bt_end'],
                        curve_mdp=curve_mdp, ts_builder=ts_builder)
print(f'  {rates.shape[0]} dates x {rates.shape[1]} contracts')
print(f'  Ladder: {list(rates.columns)}')
rates.tail(3)

In [ ]:
gap = C['spread_gap']

# Compute spread time series
spread_ts = compute_spread_curve(rates, gap=gap)
print(f'{gap_label} Spreads: {list(spread_ts.columns)}')

# Z-score, vol, roll
zscore_ts = compute_zscore_ts(spread_ts, window=C['zscore_window'])
vol_ts = spread_ts.diff().rolling(C['vol_window'], min_periods=10).std() * np.sqrt(252)

roll_ts = pd.DataFrame(np.nan, index=spread_ts.index, columns=spread_ts.columns)
for i in range(len(spread_ts)):
    row = spread_ts.iloc[i]
    for j in range(1, len(spread_ts.columns)):
        roll_ts.iloc[i, j] = row.iloc[j - 1] - row.iloc[j]
radj_ts = roll_ts / vol_ts.replace(0, np.nan)

print(f'First valid z-score: {zscore_ts.dropna(how="all").index[0]}')
print(f'Backtest starts: {C["bt_start"]}')

---
## 3. Spread Screener (Current Snapshot)

In [ ]:
snap = build_snapshot(sfr_config, rates_panel=rates)
st_key = {1: StructureType.SPD_3M, 2: StructureType.SPD_6M,
          3: StructureType.SPD_9M, 4: StructureType.SPD_12M}[gap]

if st_key in snap.structures:
    spd_data = snap.structures[st_key]
    print(f'=== {gap_label} Calendar Spread Screener ===')
    s = spd_data.summary()
    if 'peak' in s and 'trough' in s:
        print(f'  Peak: {s["peak"]["label"]} = {s["peak"]["value"]:+.1f} bp')
        print(f'  Trough: {s["trough"]["label"]} = {s["trough"]["value"]:+.1f} bp')
    display(spd_data.to_dataframe())

    # Chart
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), gridspec_kw={'height_ratios': [2, 1]})
    x = range(len(spd_data.labels))
    ax1.plot(x, spd_data.levels, 'o-', color='tab:cyan', linewidth=2, markersize=6)
    for i, (lbl, lvl) in enumerate(zip(spd_data.labels, spd_data.levels)):
        if not np.isnan(lvl):
            ax1.annotate(f'{lvl:.1f}', (i, lvl), textcoords='offset points',
                        xytext=(0, 10), ha='center', fontsize=8, color='red', fontweight='bold')
    ax1.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax1.set_xticks(x)
    ax1.set_xticklabels(spd_data.labels, rotation=45, ha='right', fontsize=9)
    ax1.set_title(f'{gap_label} Calendar Spread Curve (bp)', fontweight='bold')
    ax1.grid(True, alpha=0.3)

    colors = ['tab:green' if c >= 0 else 'tab:red' for c in spd_data.changes]
    ax2.bar(x, spd_data.changes, color=colors, alpha=0.7)
    ax2.set_xticks(x)
    ax2.set_xticklabels(spd_data.labels, rotation=45, ha='right', fontsize=9)
    ax2.set_title('1d Change (bp)', fontweight='bold')
    ax2.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## 4. Event-Driven Backtest

In [ ]:
# Build signal table
signal_table = build_spread_signal_table(spread_ts, zscore_ts, vol_ts, roll_ts, radj_ts, C)
print(f'Signal table: {len(signal_table)} dates with signals')

# Build time grid
bt_start_dt = NYC.localize(datetime.datetime.fromisoformat(C['bt_start']).replace(hour=17))
bt_end_dt = NYC.localize(datetime.datetime.now()) if C['bt_end'] == 'live' else \
            NYC.localize(datetime.datetime.fromisoformat(C['bt_end']).replace(hour=17))
bt_dates = pd.bdate_range(bt_start_dt, bt_end_dt, tz=NYC)
bt_datetimes = [d.to_pydatetime() for d in bt_dates]

# Wire triggers
entry = SFRSpreadEntryTrigger(signal_table, C)
exit_ = SFRSpreadExitTrigger(signal_table, C)
strategy = QueryStrategy(name='sfr_spread', triggers=[entry, exit_], default_mdp=curve_mdp)

# Run
bt = QueryDrivenBacktest(
    time_grid=TimeGrid(bt_datetimes),
    strategy=strategy,
    mdp=curve_mdp,
)
bt.run()

# Extract results
mtm = pd.Series(bt.mtm_history).sort_index()
daily = mtm.diff().dropna()
s = daily.std()
dd = mtm - mtm.cummax()

print(f'\n=== DEFAULT CONFIG RESULTS ===')
print(f'  Period:        {bt_start_dt.date()} to {bt_end_dt.date()} ({len(bt_datetimes)} steps)')
print(f'  Entries:       {len(bt.portfolio.trades_log)}')
print(f'  Open:          {len(bt.portfolio.positions)}')
print(f'  Final MTM:     ${mtm.iloc[-1]:+,.0f}')
print(f'  Realized P&L:  ${bt.realized_pnl:+,.0f}')
print(f'  Sharpe:        {daily.mean() / s * np.sqrt(252):.2f}' if s > 0 else '  Sharpe: N/A')
print(f'  Max DD:        ${dd.min():+,.0f}')
print(f'  Daily hit:     {(daily > 0).mean():.1%}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 12), gridspec_kw={'height_ratios': [3, 1, 1]})

ax = axes[0]
ax.plot(mtm.index, mtm.values, color='tab:cyan', linewidth=1.5)
ax.fill_between(mtm.index, 0, mtm.values, where=mtm.values >= 0, color='tab:green', alpha=0.15)
ax.fill_between(mtm.index, 0, mtm.values, where=mtm.values < 0, color='tab:red', alpha=0.15)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
sharpe_val = daily.mean() / s * np.sqrt(252) if s > 0 else 0
ax.set_title(f'{gap_label} Spread RV Backtest -- MTM ($) | Sharpe={sharpe_val:.2f}', fontweight='bold')
ax.set_ylabel('MTM ($)')
ax.grid(True, alpha=0.3)

ax = axes[1]
colors = ['tab:green' if x >= 0 else 'tab:red' for x in daily.values]
ax.bar(daily.index, daily.values, color=colors, alpha=0.6, width=1)
ax.set_title('Daily P&L ($)', fontweight='bold')
ax.grid(True, alpha=0.3)

ax = axes[2]
ax.fill_between(dd.index, 0, dd.values, color='tab:red', alpha=0.4)
ax.set_title(f'Drawdown ($) | Max DD = ${dd.min():+,.0f}', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Exit Strategy Comparison

In [ ]:
exit_configs = {
    'Mean Rev Only (44d)': {
        'exit_mean_reversion': True, 'exit_stop_loss_sd': None,
        'exit_take_profit_zscore': None, 'exit_take_profit_bp': None,
        'exit_stop_loss_bp': None, 'exit_max_holding_days': 44,
    },
    'MeanRev + Stop 2sd (22d)': {
        'exit_mean_reversion': True, 'exit_stop_loss_sd': 2.0,
        'exit_take_profit_zscore': None, 'exit_take_profit_bp': None,
        'exit_stop_loss_bp': None, 'exit_max_holding_days': 22,
    },
    'MeanRev + Stop 1.5sd (22d)': {
        'exit_mean_reversion': True, 'exit_stop_loss_sd': 1.5,
        'exit_take_profit_zscore': None, 'exit_take_profit_bp': None,
        'exit_stop_loss_bp': None, 'exit_max_holding_days': 22,
    },
    'TP z<0.5 + Stop 2sd (22d)': {
        'exit_mean_reversion': False, 'exit_stop_loss_sd': 2.0,
        'exit_take_profit_zscore': 0.5, 'exit_take_profit_bp': None,
        'exit_stop_loss_bp': None, 'exit_max_holding_days': 22,
    },
    'TP +5bp / SL -3bp (22d)': {
        'exit_mean_reversion': False, 'exit_stop_loss_sd': None,
        'exit_take_profit_zscore': None, 'exit_take_profit_bp': 5.0,
        'exit_stop_loss_bp': -3.0, 'exit_max_holding_days': 22,
    },
    'TP z<0.3 + Stop 1.5sd (15d)': {
        'exit_mean_reversion': False, 'exit_stop_loss_sd': 1.5,
        'exit_take_profit_zscore': 0.3, 'exit_take_profit_bp': None,
        'exit_stop_loss_bp': None, 'exit_max_holding_days': 15,
    },
}

results = []
fig, ax = plt.subplots(figsize=(18, 7))

for name, overrides in exit_configs.items():
    cfg = dict(C)
    cfg.update(overrides)
    try:
        st = build_spread_signal_table(spread_ts, zscore_ts, vol_ts, roll_ts, radj_ts, cfg)
        e = SFRSpreadEntryTrigger(st, cfg)
        x = SFRSpreadExitTrigger(st, cfg)
        strat = QueryStrategy(name=name, triggers=[e, x], default_mdp=curve_mdp)
        b = QueryDrivenBacktest(time_grid=TimeGrid(bt_datetimes), strategy=strat, mdp=curve_mdp, show_progress=False)
        b.run()
        m = pd.Series(b.mtm_history).sort_index()
        d = m.diff().dropna()
        sv = d.std()
        m.plot(ax=ax, label=name, linewidth=1.5)
        results.append({
            'Strategy': name,
            'MTM ($)': f'${m.iloc[-1]:+,.0f}',
            'Realized ($)': f'${b.realized_pnl:+,.0f}',
            'Entries': len(b.portfolio.trades_log),
            'Sharpe': round(d.mean() / sv * np.sqrt(252), 2) if sv > 0 else 0,
            'Max DD ($)': f'${(m - m.cummax()).min():+,.0f}',
            'Hit Rate': f'{(d > 0).mean():.1%}',
        })
    except Exception as e:
        print(f'{name}: ERROR - {e}')

ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title(f'{gap_label} Spread RV -- Exit Strategy Comparison (MTM $)', fontweight='bold')
ax.set_ylabel('MTM ($)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\n=== Exit Strategy Summary ===')
display(pd.DataFrame(results).set_index('Strategy'))

---
## 6. Carry Filter Impact

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

for carry_on in [False, True]:
    cfg = dict(C)
    cfg['entry_require_carry'] = carry_on
    st = build_spread_signal_table(spread_ts, zscore_ts, vol_ts, roll_ts, radj_ts, cfg)
    e = SFRSpreadEntryTrigger(st, cfg)
    x = SFRSpreadExitTrigger(st, cfg)
    strat = QueryStrategy(name='carry_test', triggers=[e, x], default_mdp=curve_mdp)
    b = QueryDrivenBacktest(time_grid=TimeGrid(bt_datetimes), strategy=strat, mdp=curve_mdp, show_progress=False)
    b.run()
    m = pd.Series(b.mtm_history).sort_index()
    d = m.diff().dropna()
    sv = d.std()
    sharpe = d.mean() / sv * np.sqrt(252) if sv > 0 else 0
    label = f"{'With' if carry_on else 'No'} carry filter ({len(b.portfolio.trades_log)} entries) | Sharpe={sharpe:.2f}"
    m.plot(ax=ax, label=label, linewidth=1.5)

ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Carry Filter Impact on Spread RV', fontweight='bold')
ax.set_ylabel('MTM ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Specific Spread Deep-Dive: SFR1/SFR2 (Front 3M Spread)

In [ ]:
cols = list(rates.columns)
front_label, back_label = cols[0], cols[1]
result = analyze_specific_spread(rates, front_label, back_label, sfr_config)

print(f'=== {result["trade"]} ({gap_label} Spread) ===')
for k, v in result.items():
    if k not in ('timeseries', 'zscore_ts'):
        print(f'  {k}: {v}')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), gridspec_kw={'height_ratios': [2, 1]})

ts = result['timeseries']
zs = result['zscore_ts']

ax1.plot(ts.index, ts.values, color='tab:cyan', linewidth=1.5, label=f'{result["trade"]} Level')
if result['mean'] is not None:
    ax1.axhline(result['mean'], color='tab:orange', linestyle='--', linewidth=1, alpha=0.7,
                label=f'Mean = {result["mean"]:.1f} bp')
ax1.axhline(0, color='gray', linestyle=':', linewidth=0.8)
ax1.set_title(f'{result["trade"]} Spread Level (bp)', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.fill_between(zs.index, 0, zs.values, where=zs.values >= 0, color='tab:green', alpha=0.3)
ax2.fill_between(zs.index, 0, zs.values, where=zs.values < 0, color='tab:red', alpha=0.3)
ax2.plot(zs.index, zs.values, color='tab:blue', linewidth=1)
ax2.axhline(2, color='red', linestyle='--', linewidth=0.8, alpha=0.5)
ax2.axhline(-2, color='red', linestyle='--', linewidth=0.8, alpha=0.5)
ax2.set_title('Z-Score', fontweight='bold')
ax2.set_ylim(-3.5, 3.5)
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. All Spreads: Z-Score Heatmap

In [ ]:
# Z-score for all gap sizes
zs_data = {}
for g, st_type in [(1, StructureType.SPD_3M), (2, StructureType.SPD_6M),
                    (3, StructureType.SPD_9M), (4, StructureType.SPD_12M)]:
    if st_type in snap.structures and snap.structures[st_type].labels:
        d = snap.structures[st_type]
        zs_data[STRUCTURE_LABELS[st_type]] = pd.Series(d.zscores, index=d.labels)

if zs_data:
    zs_df = pd.DataFrame(zs_data)
    fig, ax = plt.subplots(figsize=(16, 5))
    im = ax.imshow(zs_df.T.values, cmap='RdBu_r', aspect='auto', vmin=-3, vmax=3)
    ax.set_xticks(range(len(zs_df.index)))
    ax.set_xticklabels(zs_df.index, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(zs_df.columns)))
    ax.set_yticklabels(zs_df.columns)
    for i in range(len(zs_df.columns)):
        for j in range(len(zs_df.index)):
            val = zs_df.iloc[j, i]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=7,
                       color='white' if abs(val) > 1.5 else 'black')
    plt.colorbar(im, ax=ax, label='Z-Score')
    ax.set_title('Spread Z-Score Heatmap (all gaps)', fontweight='bold')
    plt.tight_layout()
    plt.show()